In [10]:
from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

MODEL = "gemini-3.5-flash-lite"

print("Gemini client initialized:", client is not None)
print("Model:", MODEL)

Gemini client initialized: True
Model: gemini-3.5-flash-lite


In [11]:
from pydantic import BaseModel, Field
from typing import Literal

class Complaint(BaseModel):
    category: Literal["PEST", "IRRIGATION", "SEED_QUALITY", "OTHER"]
    severity: int = Field(ge=1, le=5)
    district: str | None = None
    crop: str | None = None
    summary: str = Field(max_length=200)

test_inputs = [
    "Insects are eating the cotton leaves in Madurai district.",
    "Canal water has not reached our paddy field for two weeks.",
    "Half of the purchased maize seeds failed to germinate.",
    "White fungal patches are spreading on tomato leaves.",
    "The irrigation motor is running but water is not flowing.",
    "The wheat seeds received are damaged and broken.",
    "Our tractor engine is not starting.",
    "Caterpillars are damaging the brinjal crop.",
    "The irrigation pipeline has developed a major leak.",
    "The paddy crop is drying due to lack of rainfall.",
    "Aphids are attacking the chilli plants.",
    "The sunflower seeds have very poor germination.",
    "The water pump has stopped working in Salem district.",
    "The rice leaves have turned yellow due to disease.",
    "The fertilizer delivery has not arrived.",
    "Insects have damaged the groundnut crop.",
    "Water supply to the sugarcane field is insufficient.",
    "The seed packets contain poor-quality seeds.",
    "The mango crop has visible pest damage.",
    "Farm equipment was damaged during transportation."
]

print("Schema created.")
print("Total test inputs:", len(test_inputs))

Schema created.
Total test inputs: 20


Initialize Gemini

In [12]:
MODEL = "gemini-3.5-flash-lite"

In [13]:
import json
import time
import logging
from pydantic import ValidationError

logging.basicConfig(level=logging.WARNING)

# Reuse the Gemini client and MODEL from Assignment 01
print("Model:", MODEL)
print("Gemini client ready:", client is not None)

Model: gemini-3.5-flash-lite
Gemini client ready: True


Extraction function

In [14]:
def extract_complaint(text, retry=True):
    prompt = f"""
Extract the agricultural complaint details.

Return JSON with exactly these fields:
- category: PEST, IRRIGATION, SEED_QUALITY, or OTHER
- severity: integer from 1 to 5
- district: district name, or null if not mentioned
- crop: crop name, or null if not mentioned
- summary: short summary under 200 characters

Only use information present in the complaint.
Do not invent missing district or crop information.

Complaint: {text}
"""

    for attempt in range(2):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config={
                    "temperature": 0,
                    "response_mime_type": "application/json",
                    "response_schema": Complaint
                }
            )

            result = Complaint.model_validate_json(response.text)
            return result, attempt

        except (ValidationError, Exception) as e:
            if attempt == 0:
                prompt += f"\nYour previous response had this error: {e}\nCorrect it and return valid JSON."
                time.sleep(5)
            else:
                logging.warning("Extraction failed after retry: %s", e)
                return None, 2

Testing

In [15]:
result, attempt = extract_complaint(
    "Insects are damaging tomato leaves in Madurai district."
)

print(result)
print("Attempt:", attempt + 1 if result else "Failed")

category='PEST' severity=3 district='Madurai' crop='tomato' summary='Insects damaging tomato leaves in Madurai district.'
Attempt: 1


Run all 20 test inputs

In [16]:
results = []
first_success = 0
retry_success = 0
failed = 0

for i, text in enumerate(test_inputs, 1):
    result, attempt = extract_complaint(text)

    if result is not None:
        if attempt == 0:
            first_success += 1
        else:
            retry_success += 1

        results.append({
            "Input": i,
            "Category": result.category,
            "Severity": result.severity,
            "District": result.district,
            "Crop": result.crop,
            "Summary": result.summary,
            "Attempts": attempt + 1
        })
    else:
        failed += 1
        results.append({
            "Input": i,
            "Category": "FAILED",
            "Severity": None,
            "District": None,
            "Crop": None,
            "Summary": text,
            "Attempts": 2
        })

    print(f"Processed {i}/20")

import pandas as pd
display(pd.DataFrame(results))

print("\n--- Evaluation Summary ---")
print("First-attempt successes:", first_success)
print("Retry successes:", retry_success)
print("Failed after retry:", failed)

Processed 1/20
Processed 2/20
Processed 3/20
Processed 4/20
Processed 5/20
Processed 6/20
Processed 7/20
Processed 8/20
Processed 9/20
Processed 10/20
Processed 11/20
Processed 12/20
Processed 13/20
Processed 14/20
Processed 15/20
Processed 16/20
Processed 17/20
Processed 18/20


Processed 19/20
Processed 20/20


,Input,Category,Severity,District,Crop,Summary,Attempts
0,1,PEST,3.0,Madurai,cotton,Insects are eating cotton leaves in Madurai di...,1
1,2,IRRIGATION,4.0,None,paddy,Canal water has not reached the paddy field fo...,1
2,3,SEED_QUALITY,4.0,None,maize,Half of the purchased maize seeds failed to ge...,1
3,4,PEST,3.0,None,tomato,White fungal patches are spreading on tomato l...,1
4,5,IRRIGATION,3.0,None,None,The irrigation motor is running but water is n...,1
5,6,SEED_QUALITY,4.0,None,wheat,The wheat seeds received are damaged and broken.,1
6,7,OTHER,3.0,None,None,Tractor engine is not starting,1
7,8,PEST,3.0,None,brinjal,Caterpillars are damaging the brinjal crop.,1
8,9,IRRIGATION,4.0,None,None,The irrigation pipeline has developed a major ...,1
9,10,IRRIGATION,4.0,None,paddy,Paddy crop is drying due to lack of rainfall.,2



--- Evaluation Summary ---
First-attempt successes: 16
Retry successes: 3
Failed after retry: 1


retrying prompt 19

In [17]:
import time

print("Waiting for API quota to reset...")
time.sleep(35)

result, attempt = extract_complaint(test_inputs[18])

if result:
    print("SUCCESS:", result)
    print("Attempts:", attempt + 1)
    results[18] = {
        "Input": 19,
        "Category": result.category,
        "Severity": result.severity,
        "District": result.district,
        "Crop": result.crop,
        "Summary": result.summary,
        "Attempts": attempt + 1
    }
else:
    print("Still failed. Likely API quota issue.")

Waiting for API quota to reset...
SUCCESS: category='PEST' severity=3 district=None crop='mango' summary='Mango crop has visible pest damage.'
Attempts: 1


Test missing fields explicitly

In [18]:
missing_test = extract_complaint(
    "Insects are damaging the mango crop."
)

result, attempt = missing_test

print(result)

if result:
    print("\nDistrict:", result.district)
    print("Crop:", result.crop)
    print("District is None:", result.district is None)

category='PEST' severity=3 district=None crop='mango' summary='Insects are damaging the mango crop.'

District: None
Crop: mango
District is None: True


Final evaluation summary

In [19]:
import pandas as pd

results[18] = {
    "Input": 19,
    "Category": result.category,
    "Severity": result.severity,
    "District": result.district,
    "Crop": result.crop,
    "Summary": result.summary,
    "Attempts": 1
}

final_df = pd.DataFrame(results)
display(final_df)

print("\nFINAL SUMMARY")
print("Total inputs:", len(final_df))
print("Successfully extracted:", len(final_df[final_df["Category"] != "FAILED"]))
print("First-attempt successes: 16")
print("Retry / rerun successes: 4")
print("Failed after retry: 0")
print("Missing district test:", result.district is None)

,Input,Category,Severity,District,Crop,Summary,Attempts
0,1,PEST,3,Madurai,cotton,Insects are eating cotton leaves in Madurai di...,1
1,2,IRRIGATION,4,None,paddy,Canal water has not reached the paddy field fo...,1
2,3,SEED_QUALITY,4,None,maize,Half of the purchased maize seeds failed to ge...,1
3,4,PEST,3,None,tomato,White fungal patches are spreading on tomato l...,1
4,5,IRRIGATION,3,None,None,The irrigation motor is running but water is n...,1
5,6,SEED_QUALITY,4,None,wheat,The wheat seeds received are damaged and broken.,1
6,7,OTHER,3,None,None,Tractor engine is not starting,1
7,8,PEST,3,None,brinjal,Caterpillars are damaging the brinjal crop.,1
8,9,IRRIGATION,4,None,None,The irrigation pipeline has developed a major ...,1
9,10,IRRIGATION,4,None,paddy,Paddy crop is drying due to lack of rainfall.,2



FINAL SUMMARY
Total inputs: 20
Successfully extracted: 20
First-attempt successes: 16
Retry / rerun successes: 4
Failed after retry: 0
Missing district test: True
